In [1]:
!git clone https://github.com/rashibharti28/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion.git

Cloning into 'BERT-Quantization-PTQ-QAT-on-dair-ai-emotion'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 110 (delta 43), reused 37 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 8.84 MiB | 12.44 MiB/s, done.
Resolving deltas: 100% (43/43), done.
Filtering content: 100% (8/8), 1.96 GiB | 40.85 MiB/s, done.


In [2]:
# install deps
!pip install -U transformers datasets evaluate peft bitsandbytes safetensors pandas

# run inference


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 20.5 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.6.2
    Uninstalling safetensors-0.6.2:
      Successfully uninstalled safetensors-0.6.2
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pandas

In [3]:


import os, sys, tempfile, zipfile, shutil, json, traceback, time, math, gc
if "ipykernel" in sys.modules:
    sys.argv = sys.argv[:1]

import torch, numpy as np, pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from transformers import logging as hf_logging
from peft import PeftModel, PeftConfig
import evaluate
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, f1_score

hf_logging.set_verbosity_error()
os.environ["WANDB_DISABLED"] = "true"

\

MODEL_PATH = "/content/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion/QLora_Distilbert"

BATCH_SIZE = 32
MAX_LENGTH = 128
OUTPUT_CSV = "predictions.csv"
METRICS_JSON = "metrics_summary.json"
DATASET_NAME = "dair-ai/emotion"
SPLIT = "test"


def extract_if_zip(path):
    if path and os.path.isfile(path) and path.lower().endswith(".zip"):
        tmpdir = tempfile.mkdtemp(prefix="model_zip_")
        print(f"[ZIP] Extracting {path} -> {tmpdir}")
        with zipfile.ZipFile(path, "r") as z:
            z.extractall(tmpdir)
        entries = os.listdir(tmpdir)
        if len(entries) == 1 and os.path.isdir(os.path.join(tmpdir, entries[0])):
            extracted = os.path.join(tmpdir, entries[0])
        else:
            extracted = tmpdir
        print(f"[ZIP] Extracted -> {extracted}")
        return extracted, tmpdir
    return path, None

def find_model_root(base_path, max_depth=4):
    candidates = []
    for root, dirs, files in os.walk(base_path):
        rel = os.path.relpath(root, base_path)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if depth > max_depth:
            dirs[:] = []
            continue
        if "config.json" in files:
            score = 0
            if "adapter_config.json" in files:
                score += 100
            if any(w in files for w in ("pytorch_model.bin", "model.safetensors", "adapter_model.safetensors")):
                score += 50
            if any(t in files for t in ("tokenizer.json", "vocab.txt", "tokenizer_config.json")):
                score += 10
            candidates.append((score, root, depth))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[2]))
    return candidates[0][1]

def get_num_labels_and_dataset(dataset_name=DATASET_NAME):
    ds = load_dataset(dataset_name)
    if "train" in ds and hasattr(ds["train"].features["label"], "names"):
        num_labels = len(ds["train"].features["label"].names)
        return num_labels, ds
    raise RuntimeError("Can't determine number of labels from dataset.")

def load_model_and_tokenizer_auto(model_path, num_labels, no_quant=True):
    cleanup_tmp = None
    resolved, cleanup_tmp = extract_if_zip(model_path)
    if not resolved or not os.path.exists(resolved):
        if cleanup_tmp:
            shutil.rmtree(cleanup_tmp, ignore_errors=True)
        raise RuntimeError(f"Model path {model_path} not found after extraction.")
    print(f"[SEARCH] Searching for model root in: {resolved}")
    root = find_model_root(resolved)
    if root:
        print(f"[SEARCH] Candidate root: {root}")
    else:
        print("[SEARCH] No candidate root found; using provided path")
        root = resolved


    try:
        print("[LOAD] Trying direct load from:", root)
        tokenizer = None
        try:
            tokenizer = AutoTokenizer.from_pretrained(root)
        except Exception as e:
            print("[WARN] tokenizer.from_pretrained(root) failed:", e)
        model = AutoModelForSequenceClassification.from_pretrained(
            root,
            num_labels=num_labels,
            low_cpu_mem_usage=True,
            device_map=None if no_quant else ("auto" if torch.cuda.is_available() else None)
        )
        if tokenizer is None:
            try:
                tokenizer = AutoTokenizer.from_pretrained(root)
            except Exception:
                tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        print("[LOAD] Direct load succeeded.")
        return model, tokenizer, root, cleanup_tmp
    except Exception:
        print("[WARN] Direct load failed. Trace:")
        traceback.print_exc()


    adapter_root = None
    for cand in (root, resolved):
        if os.path.exists(os.path.join(cand, "adapter_config.json")):
            adapter_root = cand
            break
    if adapter_root:
        try:
            print("[LOAD] Found adapter at:", adapter_root)
            peft_cfg = PeftConfig.from_pretrained(adapter_root)
            base_name = peft_cfg.base_model_name_or_path
            print(f"[LOAD] PEFT base: {base_name}")
            try:
                tokenizer = AutoTokenizer.from_pretrained(base_name)
            except Exception:
                tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
            base_cfg = AutoConfig.from_pretrained(base_name, num_labels=num_labels)
            base_model = AutoModelForSequenceClassification.from_pretrained(
                base_name,
                config=base_cfg,
                low_cpu_mem_usage=True,
                device_map=None
            )
            model = PeftModel.from_pretrained(base_model, adapter_root)
            try:
                model = model.merge_and_unload()
            except Exception:
                pass
            print("[LOAD] PEFT adapter loaded.")
            return model, tokenizer, adapter_root, cleanup_tmp
        except Exception:
            print("[WARN] PEFT load failed. Trace:")
            traceback.print_exc()

    # Try as HF hub id/name fallback
    try:
        print("[LOAD] Trying hub load from:", model_path)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels, low_cpu_mem_usage=True)
        return model, tokenizer, model_path, cleanup_tmp
    except Exception:
        print("[WARN] Hub load failed. Trace:")
        traceback.print_exc()

    if cleanup_tmp:
        shutil.rmtree(cleanup_tmp, ignore_errors=True)
    raise RuntimeError("Failed to load model/tokenizer from MODEL_PATH. Inspect folder structure.")

def pretty_bytes(n):
    # human readable bytes
    for unit in ['B','KB','MB','GB','TB']:
        if abs(n) < 1024.0:
            return "%3.1f%s" % (n, unit)
        n /= 1024.0
    return "%.1f%s" % (n, 'PB')

def folder_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except Exception:
                pass
    return total

def model_param_stats(model):
    total = 0
    trainable = 0
    for _, p in model.named_parameters():
        num = p.numel()
        total += num
        if p.requires_grad:
            trainable += num
    return total, trainable

def read_adapter_config_if_any(model_root):
    cfg_path = os.path.join(model_root, "adapter_config.json")
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path, "r") as f:
                return json.load(f)
        except Exception:
            try:

                return PeftConfig.from_pretrained(model_root).to_dict()
            except Exception:
                return None
    else:
        # try to read any file with 'adapter' in name
        for fn in os.listdir(model_root):
            if "adapter" in fn and fn.endswith(".json"):
                try:
                    return json.load(open(os.path.join(model_root, fn)))
                except Exception:
                    continue
    return None

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

def main():
    if MODEL_PATH is None or MODEL_PATH.strip() == "":
        raise RuntimeError("Set MODEL_PATH at the top of the script.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    num_labels, ds = get_num_labels_and_dataset(DATASET_NAME)
    print("Detected num_labels =", num_labels)
    split = SPLIT if SPLIT in ds else ("validation" if "validation" in ds else "test")
    test_ds = ds[split]
    print(f"Using split: {split} (size {len(test_ds)})")

    model, tokenizer, model_root, cleanup_tmp = load_model_and_tokenizer_auto(MODEL_PATH, num_labels=num_labels, no_quant=True)
    print("Loaded model root:", model_root)

    # print model & tokenizer info
    base_model_name_or_path = getattr(model.config, "name_or_path", None)
    print("Model config name_or_path:", base_model_name_or_path)
    print("Model class:", model.__class__.__name__)
    print("Tokenizer class:", tokenizer.__class__.__name__)

    # model param stats
    total_params, trainable_params = model_param_stats(model)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,} ({100.0 * trainable_params/total_params:.4f}%)")

    # disk size if model_root exists
    if os.path.isdir(model_root):
        size_bytes = folder_size(model_root)
        print("Model folder size:", pretty_bytes(size_bytes))
    else:
        print("Model root not a folder; skipping disk-size check.")

    # read adapter/LoRA config if present
    adapter_cfg = read_adapter_config_if_any(model_root)
    if adapter_cfg:
        print("Detected adapter config (PEFT/LoRA). Adapter config contents:")
        print(json.dumps(adapter_cfg, indent=2))
    else:
        print("No adapter_config.json found (adapter/LoRA hyperparams not detected).")

    # move to device
    try:
        model.eval()
        model.to(device)
    except Exception:
        print("[WARN] model.to(device) raised (model may be sharded/quantized). Continuing.")

    # Tokenize dataset
    print("Tokenizing dataset (batched)...")
    test_tok = test_ds.map(lambda ex: tokenizer(ex["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH), batched=True, remove_columns=test_ds.column_names)
    if "label" in test_tok.column_names and "labels" not in test_tok.column_names:
        test_tok = test_tok.rename_column("label", "labels")

    # DataLoader manual loop
    test_torch = test_tok.with_format("torch")
    data_collator = DataCollatorWithPadding(tokenizer)
    dl = DataLoader(test_torch, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

    all_logits = []
    all_preds = []
    all_labels = []
    texts = []
    batch_times = []

    device_torch = torch.device(device)
    print("Starting inference loop with manual DataLoader...")
    t0 = time.time()
    for i, batch in enumerate(tqdm(dl, desc="Batches", leave=True)):
        batch_start = time.time()
        batch_for_model = {}
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch_for_model[k] = v.to(device_torch)
            else:
                batch_for_model[k] = v
        with torch.no_grad():
            outputs = model(**{k: v for k, v in batch_for_model.items() if k != "labels"})
            logits = outputs.logits
            logits_np = logits.detach().cpu().numpy()
            all_logits.append(logits_np)
            batch_preds = logits_np.argmax(axis=-1).tolist()
            all_preds.extend(batch_preds)
        if "labels" in batch_for_model:
            lbls = batch_for_model["labels"].detach().cpu().numpy().tolist()
            all_labels.extend(lbls)
        # texts from dataset
        start_idx = i * BATCH_SIZE
        end_idx = start_idx + logits_np.shape[0]
        texts.extend(test_ds["text"][start_idx:end_idx])
        batch_time = time.time() - batch_start
        batch_times.append(batch_time)
        if (i + 1) % 10 == 0:
            print(f"Processed {len(all_preds)} examples — last batch time: {batch_time:.3f}s — total elapsed: {time.time()-t0:.1f}s")
        if (i + 1) % 100 == 0 and torch.cuda.is_available():
            try:
                torch.cuda.synchronize()
                print("GPU mem alloc (MB):", torch.cuda.memory_allocated(0)/1024**2, "reserved (MB):", torch.cuda.memory_reserved(0)/1024**2)
            except Exception:
                pass
            gc.collect()

    total_time = time.time() - t0
    all_logits = np.concatenate(all_logits, axis=0) if len(all_logits) > 0 else np.array([])
    preds = np.array(all_preds)
    labels = np.array(all_labels) if len(all_labels) > 0 else None

    print(f"Inference finished. Predicted examples: {len(preds)}. Total time: {total_time:.3f}s")

    # if labels missing, fallback
    if labels is None or len(labels) == 0:
        print("[WARN] No labels found in batches; trying tokenized dataset columns...")
        if "labels" in test_tok.column_names:
            labels = np.array(test_tok["labels"])
        elif "label" in test_ds.column_names:
            labels = np.array(test_ds["label"])
        else:
            labels = None

    # align lengths
    if labels is not None:
        if len(preds) != len(labels):
            L = min(len(preds), len(labels))
            print(f"[WARN] preds ({len(preds)}) != labels ({len(labels)}). Truncating to {L}.")
            preds = preds[:L]
            labels = labels[:L]
            texts = texts[:L]
            all_logits = all_logits[:L]

    # compute latency stats
    per_example_times = []
    # derive per-sample times from batch_times and batch sizes
    # assume last batch size = logits_np.shape[0] stored — use test_ds size to compute exact sizes
    n_examples = len(preds)
    # distribute batch times according to BATCH_SIZE except maybe last
    sizes = []
    n = n_examples
    full_batches = n // BATCH_SIZE
    for _ in range(full_batches):
        sizes.append(BATCH_SIZE)
    rem = n % BATCH_SIZE
    if rem:
        sizes.append(rem)
    # If batch_times length differs from sizes, adjust
    bt = batch_times[:len(sizes)]
    if len(bt) != len(sizes):
        # fallback: uniform average
        avg_batch = sum(batch_times)/len(batch_times) if len(batch_times)>0 else total_time/(n_examples+1e-9)
        bt = [avg_batch] * len(sizes)
    for b_t, s in zip(bt, sizes):
        per_example_times.extend([b_t / s] * s)
    per_example_times = np.array(per_example_times)
    if len(per_example_times) > 0:
        mean_ms = per_example_times.mean() * 1000.0
        median_ms = np.median(per_example_times) * 1000.0
        p90 = np.percentile(per_example_times, 90) * 1000.0
        p95 = np.percentile(per_example_times, 95) * 1000.0
        p99 = np.percentile(per_example_times, 99) * 1000.0
    else:
        mean_ms = median_ms = p90 = p95 = p99 = None

    # compute metrics if labels exist
    summary = {
        "model_root": model_root,
        "device": device,
        "num_labels": int(num_labels),
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "trainable_percent": float(100.0 * trainable_params / total_params) if total_params>0 else None,
        "model_folder_size_bytes": int(size_bytes) if os.path.isdir(model_root) else None,
        "latency_mean_ms": mean_ms,
        "latency_median_ms": median_ms,
        "latency_p90_ms": p90,
        "latency_p95_ms": p95,
        "latency_p99_ms": p99,
        "total_inference_time_s": float(total_time),
        "batch_times_s": [float(x) for x in batch_times]
    }

    if labels is None:
        print("[ERROR] No reference labels available — cannot compute supervised metrics. Saving predictions only.")
        rows = []
        for i, txt in enumerate(texts):
            prob_str = json.dumps([float(x) for x in all_logits[i].tolist()]) if i < len(all_logits) else "[]"
            rows.append({"text": txt, "pred_label": int(preds[i]) if i < len(preds) else None, "pred_probs": prob_str})
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
        print("Saved predictions to", OUTPUT_CSV)
    else:
        # classification report & confusion matrix
        label_names = test_ds.features["label"].names if hasattr(test_ds.features["label"], "names") else [str(i) for i in range(num_labels)]
        report = classification_report(labels, preds, target_names=label_names, digits=4, output_dict=True)
        cm = confusion_matrix(labels, preds)
        # compute macro/micro/weighted
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro")
        precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds, average="micro")
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, preds, average="weighted")
        acc_val = accuracy_score(labels, preds)

        # print summary
        print("\n=== METRICS SUMMARY ===")
        print(f"Accuracy: {acc_val:.4f}")
        print(f"Macro Precision/Recall/F1: {precision_macro:.4f} / {recall_macro:.4f} / {f1_macro:.4f}")
        print(f"Micro Precision/Recall/F1: {precision_micro:.4f} / {recall_micro:.4f} / {f1_micro:.4f}")
        print(f"Weighted Precision/Recall/F1: {precision_weighted:.4f} / {recall_weighted:.4f} / {f1_weighted:.4f}")
        print("\nPer-class classification report:")
        print(classification_report(labels, preds, target_names=label_names, digits=4))
        print("Confusion matrix (rows=true, cols=pred):")
        print(pd.DataFrame(cm, index=label_names, columns=label_names))
        # add to summary
        summary.update({
            "accuracy": float(acc_val),
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "f1_macro": float(f1_macro),
            "precision_micro": float(precision_micro),
            "recall_micro": float(recall_micro),
            "f1_micro": float(f1_micro),
            "precision_weighted": float(precision_weighted),
            "recall_weighted": float(recall_weighted),
            "f1_weighted": float(f1_weighted),
            "classification_report": report,
            "confusion_matrix": cm.tolist()
        })

        # Save CSV of predictions aligned
        rows = []
        L = min(len(texts), len(preds))
        for i in range(L):
            true_lab = label_names[int(labels[i])] if label_names else int(labels[i])
            pred_lab = label_names[int(preds[i])] if label_names else int(preds[i])
            rows.append({
                "text": texts[i],
                "label": true_lab,
                "pred_label": pred_lab,
                "pred_probs": json.dumps([float(x) for x in all_logits[i].tolist()])
            })
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
        print("Saved predictions to", OUTPUT_CSV)

    # add adapter config info to summary if present
    if adapter_cfg is not None:
        summary["adapter_config"] = adapter_cfg

    # LoRA/adapter hyperparams summary if available in adapter_cfg
    if adapter_cfg:
        # common keys: r, lora_alpha, lora_dropout, bias, target_modules etc.
        for k in ("r", "lora_alpha", "lora_dropout", "bias", "target_modules", "task_type"):
            if k in adapter_cfg:
                summary[f"adapter_{k}"] = adapter_cfg[k]

    # model size & params
    summary["total_params"] = int(total_params)
    summary["trainable_params"] = int(trainable_params)
    try:
        summary["model_folder_size_bytes"] = int(folder_size(model_root)) if os.path.isdir(model_root) else None
    except Exception:
        pass

    # save metrics summary json
    try:
        save_json(METRICS_JSON, summary)
        print("Saved metrics summary to", METRICS_JSON)
    except Exception:
        print("[WARN] Failed to save metrics JSON.")

    # cleanup tmp extraction
    if cleanup_tmp:
        try:
            print("[CLEANUP] removing", cleanup_tmp)
            shutil.rmtree(cleanup_tmp, ignore_errors=True)
        except Exception:
            pass

if __name__ == "__main__":
    main()


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

...

[LOAD] Direct load succeeded.
Loaded model root: /content/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion/QLora_Distilbert
Model config name_or_path: distilbert-base-uncased
Model class: DistilBertForSequenceClassification
Tokenizer class: DistilBertTokenizerFast
Total params: 68,143,116
Trainable params: 0 (0.0000%)
Model folder size: 20.0MB
Detected adapter config (PEFT/LoRA). Adapter config contents:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "distilbert-base-uncased",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "megatron_config": null,
  "megatron_core": "megatr

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Starting inference loop with manual DataLoader...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Processed 320 examples — last batch time: 0.094s — total elapsed: 1.7s
Processed 640 examples — last batch time: 0.094s — total elapsed: 2.8s
Processed 960 examples — last batch time: 0.093s — total elapsed: 3.8s
Processed 1280 examples — last batch time: 0.094s — total elapsed: 4.9s
Processed 1600 examples — last batch time: 0.095s — total elapsed: 5.9s
Processed 1920 examples — last batch time: 0.096s — total elapsed: 6.9s
Inference finished. Predicted examples: 2000. Total time: 7.254s
[WARN] No labels found in batches; trying tokenized dataset columns...

=== METRICS SUMMARY ===
Accuracy: 0.9165
Macro Precision/Recall/F1: 0.8677 / 0.8776 / 0.8724
Micro Precision/Recall/F1: 0.9165 / 0.9165 / 0.9165
Weighted Precision/Recall/F1: 0.9176 / 0.9165 / 0.9169

Per-class classification report:
              precision    recall  f1-score   support

     sadness     0.9519    0.9535    0.9527       581
         joy     0.9471    0.9281    0.9375       695
        love     0.7965    0.8616    